# ezjev 评测（Colab）· 第 2 步：跑完整 Decision Index 并提交

流程：重建评测集 → 用 vLLM 加 llm2jev 起一个 `/v1/systemone` 服务 → 先跑 100 条冒烟 → 跑完整的 150,317 条请求 → 上传结果 → 提 PR。

**需要准备**

1. Colab Pro，运行时选 **A100**；
2. Secrets 里的 `HF_TOKEN`（和第 1 步同一个）；
3. 用这个 HF 账号**先接受 [HLE 数据集](https://huggingface.co/datasets/cais/hle) 的使用条款**（评测集里有 HLE，不接受的话重建会失败）；
4. Google Drive 留出约 10 GB。

**耗时**：重建评测集约 1 小时（只需要做一次，结果存在 Drive 上），完整评测预计 5–8 小时。
runner 是一条一条顺序发请求的，**支持断点续跑**：断线后重新按顺序运行全部单元格，已经完成的请求会跳过。

**规则提醒**（kit 里已经强制执行）：不截断、不删选项、不按 benchmark 调提示词、没回答的按错算。
上下文装不下的请求会记为 `unsupported`（算错），所以 `MAX_MODEL_LEN` 尽量开大。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 安装：评测 kit（从 GitHub 拿源码，kit 里需要 hub/ 目录）、vLLM、llm2jev
!git clone -q https://github.com/apolinario/decision-index /content/decision-index 2>/dev/null || git -C /content/decision-index pull -q
!pip -q install -e "/content/decision-index[rebuild]"
!pip -q install -U "vllm>=0.30" llm2jev

In [ ]:
# ===== 配置 =====
import os, json, subprocess, time, requests
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import whoami, hf_hub_download
HF_USER = whoami()["name"]

HF_REPO   = f"{HF_USER}/ezjev-4b"            # 第 1 步上传的模型
RUN_NAME  = "ezjev-4b"
DI_DIR    = "/content/drive/MyDrive/decision-index"   # 评测集和结果放在 Drive 上
SUITE_DIR = f"{DI_DIR}/suite-0.2"
RUN_DIR   = f"{DI_DIR}/runs/{RUN_NAME}"
RESULTS_REPO = f"{HF_USER}/decision-index-results-{RUN_NAME}"   # 结果上传到这个 dataset
MAX_MODEL_LEN = 131072   # 越大越少 unsupported；A100 40GB 上如果 vLLM 启动 OOM，就降到 65536

TEMPERATURE = json.load(open(hf_hub_download(HF_REPO, "ezjev.json")))["temperature"]
os.makedirs(f"{DI_DIR}/runs", exist_ok=True)
print(HF_REPO, "T =", TEMPERATURE)

## 1. 重建评测集（只做一次）

评测集不公开分发，要从各个公开来源重建。会下载约 7 GB，临时文件放在 Colab 本地磁盘，最后只把结果文件存到 Drive。
kit 会校验哈希，和官方逐字节一致才算成功。

In [ ]:
if os.path.exists(f"{SUITE_DIR}/manifest.json"):
    print("评测集已存在:", SUITE_DIR)
else:
    %cd /content/decision-index
    !python -m decision_index suite rebuild --work /content/di-work
    BUILT = "/content/di-work/artifacts/benchmark-suite/release-v2-rebuilt"
    !python -m decision_index suite import --dir {SUITE_DIR} --rows {BUILT}/selected-rows.jsonl.gz --added-rows {BUILT}/added-rows.jsonl.gz
!ls -la {SUITE_DIR}

## 2. 启动服务：vLLM（算 logprob）+ llm2jev（`/v1/systemone`）

两个服务都在后台跑，日志写在 `/content/vllm.log` 和 `/content/llm2jev.log`。断线重连后要重新运行这个单元格。

In [ ]:
def wait(url, name, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, timeout=5).status_code == 200:
                print(name, "就绪", flush=True); return
        except requests.RequestException:
            pass
        time.sleep(10)
    raise RuntimeError(f"{name} 没起来，看日志")

subprocess.run("pkill -f 'vllm serve' ; pkill -f llm2jev", shell=True)
vllm = subprocess.Popen(
    f"vllm serve {HF_REPO} --port 8000 --max-logprobs 256 --return-tokens-as-token-ids "
    f"--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization 0.90 > /content/vllm.log 2>&1", shell=True)
wait("http://127.0.0.1:8000/health", "vLLM")
jevsrv = subprocess.Popen(
    f"llm2jev --model {HF_REPO} --backend vllm --url http://127.0.0.1:8000 --port 8080 "
    f"--temperature {TEMPERATURE} > /content/llm2jev.log 2>&1", shell=True)
time.sleep(30)
!tail -n 3 /content/llm2jev.log

In [ ]:
# 手动问一个问题，确认服务正常
r = requests.post("http://127.0.0.1:8080/v1/systemone", json={
    "state": [{"role": "user", "content": "I was charged twice. Please refund the duplicate."}],
    "questions": {
        "refund": {"type": "noul", "instructions": "Does the user request a refund?"},
        "department": {"type": "choice", "instructions": "Which department should handle this?",
                       "criteria": {"billing": "Payments and refunds", "technical": "Software bugs"}}}})
print(json.dumps(r.json(), indent=1))

## 3. 冒烟测试：随机 100 条

确认没有报错，并看一下单条延迟。榜单要求在 RTX PRO 6000 上**中位延迟低于 1000 ms**；4B 模型通常只要几十到几百毫秒。

In [ ]:
%cd /content/decision-index
!python -m decision_index suite sample --dir {SUITE_DIR} --n 100 --out /content/sample-100.jsonl.gz
!python -m decision_index run --engine http --option base_url=http://127.0.0.1:8080 --option model={RUN_NAME} --suite-dir {SUITE_DIR} --rows /content/sample-100.jsonl.gz --out /content/runs/sample-100 --fresh
!python -m decision_index score --suite-dir {SUITE_DIR} --results /content/runs/sample-100/results.jsonl --out /content/runs/sample-100
import collections, statistics
rows = [json.loads(l) for l in open("/content/runs/sample-100/results.jsonl")]
print(collections.Counter(r["status"] for r in rows))
print("中位延迟 ms:", statistics.median(r["total_wall_ms"] for r in rows if r["status"] == "ok"))
for r in rows:
    if r["status"] != "ok":
        print(r["dataset"], r["status"], r.get("error", "")[:200])

## 4. 完整评测（可断点续跑）

在后台运行，进度写进 Drive 上的 `results.jsonl`。`--compact` 不保存题目原文，这样结果可以公开上传，不违反评测集的授权条款。
跑完会自动打分，生成 `scores.json`。

**断线之后**：重新运行「配置」和「启动服务」两个单元格，再运行下面这个单元格，会从断点继续。

In [ ]:
%cd /content/decision-index
full = subprocess.Popen(
    f"python -m decision_index pipeline --engine http --option base_url=http://127.0.0.1:8080 "
    f"--option model={RUN_NAME} --suite-dir {SUITE_DIR} --out {RUN_DIR} --compact "
    f">> {DI_DIR}/{RUN_NAME}.log 2>&1", shell=True)
print("已在后台启动，用下一个单元格看进度")

In [ ]:
# 看进度（随时可以重复运行）
import collections
TOTAL = 119898 + 30419
st = collections.Counter()
if os.path.exists(f"{RUN_DIR}/results.jsonl"):
    for l in open(f"{RUN_DIR}/results.jsonl"):
        st[json.loads(l)["status"]] += 1
done = sum(st.values())
print(f"{done}/{TOTAL} ({100 * done / TOTAL:.1f}%)", dict(st))
print("后台进程:", "运行中" if full.poll() is None else f"已结束 (exit {full.returncode})")
!tail -n 5 {DI_DIR}/{RUN_NAME}.log

In [ ]:
# 跑完后看分数（如果 pipeline 中途断过，也可以手动打一次分）
if not os.path.exists(f"{RUN_DIR}/scores.json"):
    !cd /content/decision-index && python -m decision_index score --suite-dir {SUITE_DIR} --results {RUN_DIR}/results.jsonl --out {RUN_DIR}
s = json.load(open(f"{RUN_DIR}/scores.json"))
idx = json.load(open(f"{RUN_DIR}/index.json"))
print("complete:", s.get("complete"))
print("Decision Index 0.2.1:", idx.get("index"), " raw:", idx.get("raw_index"))
for a, v in idx.get("areas", {}).items():
    print(f"  {a:10s} skill {v.get('skill')}  coverage {v.get('coverage')}")

## 5. 上传结果并提交

1. 运行下面的单元格，把 `runs/ezjev-4b/` 上传到一个 HF dataset；
2. 在 Hugging Face 上把**模型仓库改成公开**；
3. fork [apolinario/decision-index](https://github.com/apolinario/decision-index)，在 `submissions/README.md`（没有就新建）加一行，然后提 PR：

```
| ezjev-4b | https://huggingface.co/<你>/ezjev-4b | https://huggingface.co/datasets/<你>/decision-index-results-ezjev-4b (runs/ezjev-4b/scores.json) | http engine → llm2jev 0.6.x over vLLM, temperature <T> | Colab A100 |
```

在 PR 描述里写清楚：底座是 Qwen3.5-4B，LoRA 合并；只用了公开数据集的 train split，没用任何评测集数据；
服务命令照抄模型卡；`MAX_MODEL_LEN` 设成多少（这算声明的容量上限，可以接受，只要没有截断）。
维护者会在 RTX PRO 6000 上自己测延迟、抽检答案，然后重新打分。

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(RESULTS_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=RUN_DIR, path_in_repo=f"runs/{RUN_NAME}", repo_id=RESULTS_REPO, repo_type="dataset",
                  commit_message=f"Decision Index 0.2.1 run: {RUN_NAME}")
print(f"https://huggingface.co/datasets/{RESULTS_REPO}  （现在是私有的；提 PR 前改成公开）")